# Generacion de Embeddings CLIP

**openai/clip-vit-base-patch32** — dim 512 — 8 datasets pre-validados

## Pasos
1. **Runtime → Change runtime type → GPU (T4)**
2. **Runtime → Run all**
3. Descargar `clip_embeddings.zip` al final
4. En tu PC, descomprimir en `clustering-python/embeddings/`

## Datasets

| # | Dataset | Tipo | Clases | N aprox | Fuente |
|---|---|---|---|---|---|
| 1 | 20NG (5 clases) | Texto | 5 | ~750-900/clase | sklearn |
| 2 | BBC News | Texto | 5 | 2225 | HuggingFace SetFit/bbc-news |
| 3 | 20NG (2 clases) | Texto | 2 | ~1000 | sklearn |
| 4 | 20NG (3 clases) | Texto | 3 | ~1500 | sklearn |
| 5 | AG News (4k) | Texto | 4 | 4000 | HuggingFace ag_news |
| 6 | ORL Faces | Imagen | 40 | 400 | sklearn fetch_olivetti_faces |
| 7 | MNIST (5k) | Imagen | 10 | 5000 | torchvision |
| 8 | Fashion-MNIST (5k) | Imagen | 10 | 5000 | torchvision |

In [ ]:
# CELDA 1: Instalar dependencias
# Nota: el error 'fsspec incompatible' que puede aparecer es un WARNING
# inofensivo de gcsfs (pre-instalado en Colab). No afecta la ejecucion.
!pip install -q --upgrade datasets huggingface_hub
!pip install -q transformers torchvision scikit-learn pillow

In [ ]:
# CELDA 2: Imports y verificacion de GPU
import os, time, zipfile
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('ADVERTENCIA: No hay GPU. El proceso sera mas lento pero funcionara.')

In [ ]:
# CELDA 3: Cargar modelo CLIP
MODEL_NAME = 'openai/clip-vit-base-patch32'
print('Cargando', MODEL_NAME, '...')
model = CLIPModel.from_pretrained(MODEL_NAME).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
EMBED_DIM = model.config.projection_dim
print('Modelo listo. Dimension de embedding:', EMBED_DIM)

In [ ]:
# CELDA 4: Funciones de embedding (texto e imagen)
@torch.inference_mode()
def embed_texts(texts, batch_size=64):
    """Embeddings de texto. Devuelve (N, 512) float32 L2-normalizados."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        inputs = processor(
            text=batch, return_tensors='pt',
            padding=True, truncation=True, max_length=77
        ).to(device)
        emb = model.get_text_features(**inputs)
        emb = torch.nn.functional.normalize(emb, p=2, dim=-1)
        all_embs.append(emb.cpu().numpy().astype(np.float32))
        if (i // batch_size) % 10 == 0:
            print(f'  Procesados {min(i+batch_size, len(texts))}/{len(texts)}', end='\r')
    return np.concatenate(all_embs, axis=0)


@torch.inference_mode()
def embed_images(images, batch_size=64):
    """Embeddings de imagen. Acepta lista de PIL Images."""
    all_embs = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        batch = [img.convert('RGB') if img.mode != 'RGB' else img for img in batch]
        inputs = processor(images=batch, return_tensors='pt').to(device)
        emb = model.get_image_features(**inputs)
        emb = torch.nn.functional.normalize(emb, p=2, dim=-1)
        all_embs.append(emb.cpu().numpy().astype(np.float32))
        if (i // batch_size) % 10 == 0:
            print(f'  Procesadas {min(i+batch_size, len(images))}/{len(images)} imagenes', end='\r')
    return np.concatenate(all_embs, axis=0)

In [ ]:
# CELDA 5: Helper para guardar .npz
OUT_DIR = Path('embeddings/clip')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_npz(name, X, y, class_names, source):
    """Guarda embeddings como .npz con metadata."""
    out_path = OUT_DIR / (name + '.npz')
    metadata = {
        'source': source, 'model': MODEL_NAME,
        'dim': int(X.shape[1]), 'n': int(X.shape[0]),
        'n_classes': int(len(np.unique(y))),
    }
    np.savez_compressed(
        out_path,
        X=X.astype(np.float32),
        y=np.asarray(y, dtype=np.int64),
        class_names=np.asarray(class_names, dtype=object),
        metadata=np.asarray([metadata], dtype=object),
    )
    size_mb = out_path.stat().st_size / 1024**2
    # Verificar que la normalizacion L2 es correcta
    norms = np.linalg.norm(X, axis=1)
    norm_ok = np.allclose(norms, 1.0, atol=1e-4)
    print(f'  [OK] {name:28s} N={metadata["n"]:5d} K={metadata["n_classes"]:3d} {size_mb:.2f}MB L2={"OK" if norm_ok else "FALLO"}')

## TEXTO - Datasets 1 a 5

In [ ]:
# [1/8] 20 Newsgroups (5 clases) - reemplaza BBC Sport
# BBC Sport fue removido de HuggingFace. Usamos 20NG con 5 categorias
# tematicas distintas, tamano similar (~750-900 por clase).
from sklearn.datasets import fetch_20newsgroups

print('\n[1/8] 20 Newsgroups (5 clases)')
t0 = time.time()
cats_5 = [
    'rec.sport.hockey',
    'rec.sport.baseball',
    'sci.med',
    'sci.space',
    'talk.politics.misc',
]
ds = fetch_20newsgroups(
    subset='all', categories=cats_5,
    remove=('headers', 'footers', 'quotes')
)
texts = ds.data
labels = list(ds.target)
class_names = list(ds.target_names)
print(f'  Distribucion de clases: {dict(zip(class_names, [labels.count(i) for i in range(len(class_names))]))}')
X = embed_texts(texts)
save_npz('20ng_5classes', X, labels, class_names, 'sklearn:fetch_20newsgroups (5 cats)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [2/8] BBC News - SetFit/bbc-news (confirmado activo en HF)
# 2225 articulos, 5 categorias: business, entertainment, politics, sport, tech
from datasets import load_dataset

print('\n[2/8] BBC News (SetFit/bbc-news)')
t0 = time.time()

# Cargar train; intentar tambien test para tener todos los datos
ds_train = load_dataset('SetFit/bbc-news', split='train')
try:
    ds_test = load_dataset('SetFit/bbc-news', split='test')
    from datasets import concatenate_datasets
    ds = concatenate_datasets([ds_train, ds_test])
except Exception:
    ds = ds_train
    print('  (Solo split train disponible)')

texts = list(ds['text'])
labels = list(ds['label'])

# Construir lista de nombres de clase ordenada por id
if 'label_text' in ds.column_names:
    label_to_text = {}
    for lid, ltxt in zip(ds['label'], ds['label_text']):
        label_to_text[lid] = ltxt
    class_names = [label_to_text[i] for i in sorted(label_to_text.keys())]
else:
    class_names = [str(c) for c in sorted(set(labels))]

print(f'  Clases: {class_names}')
X = embed_texts(texts)
save_npz('bbc_news', X, labels, class_names, 'huggingface:SetFit/bbc-news')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [3/8] 20 Newsgroups (2 clases)
print('\n[3/8] 20 Newsgroups (2 clases)')
t0 = time.time()
cats_2 = ['alt.atheism', 'soc.religion.christian']
ds = fetch_20newsgroups(
    subset='all', categories=cats_2,
    remove=('headers', 'footers', 'quotes')
)
texts, labels = ds.data, list(ds.target)
class_names = list(ds.target_names)
X = embed_texts(texts)
save_npz('20ng_2classes', X, labels, class_names, 'sklearn:fetch_20newsgroups')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [4/8] 20 Newsgroups (3 clases)
print('\n[4/8] 20 Newsgroups (3 clases)')
t0 = time.time()
cats_3 = ['comp.graphics', 'rec.sport.hockey', 'sci.med']
ds = fetch_20newsgroups(
    subset='all', categories=cats_3,
    remove=('headers', 'footers', 'quotes')
)
texts, labels = ds.data, list(ds.target)
class_names = list(ds.target_names)
X = embed_texts(texts)
save_npz('20ng_3classes', X, labels, class_names, 'sklearn:fetch_20newsgroups')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [5/8] AG News (subsample 1000 por clase = 4000 total)
print('\n[5/8] AG News (4k subsample)')
t0 = time.time()
ds_ag = load_dataset('ag_news', split='train')
ds_ag = ds_ag.shuffle(seed=42)

PER_CLASS = 1000
selected_idx = []
counts = {0: 0, 1: 0, 2: 0, 3: 0}
for i, label in enumerate(ds_ag['label']):
    if counts[label] < PER_CLASS:
        selected_idx.append(i)
        counts[label] += 1
    if all(c >= PER_CLASS for c in counts.values()):
        break

ds_sub = ds_ag.select(selected_idx)
texts = list(ds_sub['text'])
labels = list(ds_sub['label'])
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']
print(f'  Distribucion: {dict(zip(class_names, [labels.count(i) for i in range(4)]))}')  
X = embed_texts(texts)
save_npz('ag_news_4k', X, labels, class_names, 'huggingface:ag_news (1k/class)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

## IMAGENES - Datasets 6 a 8

In [ ]:
# [6/8] ORL Faces (Olivetti)
from sklearn.datasets import fetch_olivetti_faces

print('\n[6/8] ORL Faces (Olivetti)')
t0 = time.time()
ds_oli = fetch_olivetti_faces(shuffle=False)

images = []
for img_arr in ds_oli.images:  # 64x64 grayscale float
    img = (img_arr * 255).astype(np.uint8)
    pil = Image.fromarray(img, mode='L').convert('RGB')
    images.append(pil)

labels = list(ds_oli.target)
class_names = [f'person_{i}' for i in range(40)]
X = embed_images(images)
save_npz('orl_faces', X, labels, class_names, 'sklearn:fetch_olivetti_faces')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [7/8] MNIST (subsample 500/clase = 5000 total)
import torchvision

print('\n[7/8] MNIST (5k subsample)')
t0 = time.time()
mnist = torchvision.datasets.MNIST(
    root='/tmp/mnist', train=True, download=True
)

PER_CLASS = 500
targets = np.asarray(mnist.targets)
selected_idx = []
for c in range(10):
    idx_c = np.where(targets == c)[0][:PER_CLASS]
    selected_idx.extend(idx_c.tolist())

images, labels = [], []
for i in selected_idx:
    img, lab = mnist[i]
    images.append(img.convert('RGB'))
    labels.append(int(lab))

class_names = [str(i) for i in range(10)]
X = embed_images(images)
save_npz('mnist_5k', X, labels, class_names, 'torchvision:MNIST (500/class)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [8/8] Fashion-MNIST (subsample 500/clase = 5000 total)
print('\n[8/8] Fashion-MNIST (5k subsample)')
t0 = time.time()
fmnist = torchvision.datasets.FashionMNIST(
    root='/tmp/fmnist', train=True, download=True
)

PER_CLASS = 500
targets = np.asarray(fmnist.targets)
selected_idx = []
for c in range(10):
    idx_c = np.where(targets == c)[0][:PER_CLASS]
    selected_idx.extend(idx_c.tolist())

images, labels = [], []
for i in selected_idx:
    img, lab = fmnist[i]
    images.append(img.convert('RGB'))
    labels.append(int(lab))

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
              'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle_boot']
X = embed_images(images)
save_npz('fashion_mnist_5k', X, labels, class_names, 'torchvision:FashionMNIST (500/class)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

## Resumen y descarga

In [ ]:
# Resumen de todo lo generado
print('\n=== RESUMEN FINAL ===')
print(f'{"Dataset":<28} {"N":>6} {"D":>5} {"K":>4} {"MB":>7} {"L2":>4}')
print('-' * 58)
total_mb = 0
all_ok = True
for f in sorted(OUT_DIR.glob('*.npz')):
    size_mb = f.stat().st_size / 1024**2
    total_mb += size_mb
    npz = np.load(f, allow_pickle=True)
    X, y = npz['X'], npz['y']
    norms = np.linalg.norm(X, axis=1)
    norm_ok = np.allclose(norms, 1.0, atol=1e-4)
    if not norm_ok:
        all_ok = False
    print(f'{f.stem:<28} {X.shape[0]:>6} {X.shape[1]:>5} {len(np.unique(y)):>4} {size_mb:>7.2f} {"OK" if norm_ok else "FALLO":>4}')

print('-' * 58)
print(f'{"TOTAL":28} {"":>6} {"":>5} {"":>4} {total_mb:>7.2f}')
print()
if all_ok:
    print('✓ Todos los archivos pasaron verificacion L2.')
else:
    print('⚠ Algunos archivos tienen problemas de normalizacion.')

In [ ]:
# Crear ZIP para descargar
ZIP_PATH = 'clip_embeddings.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUT_DIR.glob('*.npz'):
        # Guardamos con ruta relativa embeddings/clip/<file>.npz
        # Al descomprimir en el proyecto queda en la carpeta correcta
        zf.write(f, arcname=f'embeddings/clip/{f.name}')

zip_mb = os.path.getsize(ZIP_PATH) / 1024**2
print(f'ZIP creado: {ZIP_PATH} ({zip_mb:.2f} MB)')
print('Ejecuta la siguiente celda para descargarlo.')

In [ ]:
# Descargar al navegador
from google.colab import files
files.download(ZIP_PATH)

In [ ]:
# (OPCIONAL) Guardar en Google Drive
# Si la descarga directa falla (archivos grandes), descomentar esto:

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(ZIP_PATH, '/content/drive/MyDrive/clip_embeddings.zip')
# print('Copiado a Google Drive en MyDrive/clip_embeddings.zip')

## Pasos siguientes en tu PC

1. Descomprimir `clip_embeddings.zip` **en la raiz del proyecto** `clustering-python/`
   ```
   # Windows: clic derecho → Extraer aqui
   # El ZIP ya incluye la ruta embeddings/clip/ adentro
   ```

2. Verificar la estructura resultante:
   ```
   clustering-python/
   └── embeddings/
       └── clip/
           ├── 20ng_2classes.npz
           ├── 20ng_3classes.npz
           ├── 20ng_5classes.npz
           ├── ag_news_4k.npz
           ├── bbc_news.npz
           ├── fashion_mnist_5k.npz
           ├── mnist_5k.npz
           └── orl_faces.npz
   ```

3. Probar que carga bien:
   ```cmd
   venv\Scripts\activate
   python embeddings_loader.py clip
   ```

4. Correr los 8 algoritmos en paralelo:
   ```cmd
   python testing.py 9 --model clip
   ```